# FLAME Dataset Exploration

Exploring the FLAME 2 dataset for EmberEye fire detection.

**Dataset:** FLAME 2 - Fire detection and modeLing: Aerial Multi-spectral imagE dataset
- Source: [IEEE DataPort](https://ieee-dataport.org/open-access/flame-2-fire-detection-and-modeling-aerial-multi-spectral-image-dataset)
- Contains: RGB + Thermal paired imagery from UAV
- Classes: Fire, Smoke, No-fire

## Goals
1. Understand dataset structure
2. Visualize sample images (RGB + thermal pairs)
3. Analyze class distribution
4. Prepare for YOLOv11 training format

In [ ]:
# Standard imports
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Set paths - update after downloading dataset
DATA_DIR = Path("../data")
FLAME_DIR = DATA_DIR / "flame2"  # Update to actual path after download

print(f"Looking for data in: {FLAME_DIR.resolve()}")

## 1. Dataset Structure

After downloading, explore what we have:

In [ ]:
# List top-level contents
if FLAME_DIR.exists():
    for item in sorted(FLAME_DIR.iterdir()):
        if item.is_dir():
            count = len(list(item.glob("*")))
            print(f"📁 {item.name}/ ({count} items)")
        else:
            print(f"📄 {item.name} ({item.stat().st_size / 1024:.1f} KB)")
else:
    print(f"⚠️  Dataset not found at {FLAME_DIR}")
    print("Download from: https://ieee-dataport.org/open-access/flame-2-fire-detection-and-modeling-aerial-multi-spectral-image-dataset")

## 2. Sample Visualization

View RGB and thermal pairs side by side:

In [ ]:
def show_image_pair(rgb_path: Path, thermal_path: Path, title: str = ""):
    """Display RGB and thermal images side by side."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # RGB
    rgb_img = Image.open(rgb_path)
    axes[0].imshow(rgb_img)
    axes[0].set_title(f"RGB: {rgb_path.name}")
    axes[0].axis("off")
    
    # Thermal
    thermal_img = Image.open(thermal_path)
    axes[1].imshow(thermal_img, cmap="hot")  # hot colormap for thermal
    axes[1].set_title(f"Thermal: {thermal_path.name}")
    axes[1].axis("off")
    
    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

# TODO: Update paths after exploring dataset structure
# show_image_pair(FLAME_DIR / "rgb" / "sample.jpg", FLAME_DIR / "thermal" / "sample.jpg", "Fire Sample")

## 3. Class Distribution

Count images per class:

In [ ]:
def count_by_class(base_dir: Path) -> dict:
    """Count images in each class subdirectory."""
    counts = {}
    for class_dir in sorted(base_dir.iterdir()):
        if class_dir.is_dir():
            n_images = len(list(class_dir.glob("*.jpg"))) + len(list(class_dir.glob("*.png")))
            counts[class_dir.name] = n_images
    return counts

# TODO: Run after downloading
# class_counts = count_by_class(FLAME_DIR / "images")
# print("Class distribution:")
# for cls, count in class_counts.items():
#     print(f"  {cls}: {count}")

## 4. Annotation Format Check

EmberEye trains YOLO11-seg (segmentation), so its labels are outlines, one line per fire region:
```
class_id x1 y1 x2 y2 ... xn yn
```
with every coordinate divided by the image width or height (0 to 1). FLAME 2 itself ships only frame-level labels (`labels.txt`), so the outlines are made from the thermal frames.

Check what annotation files, if any, the download contains:

In [ ]:
# TODO: Inspect annotation files after download
# Look for: .txt (YOLO), .xml (Pascal VOC), .json (COCO)

def find_annotation_files(base_dir: Path) -> dict:
    """Find annotation files by extension."""
    extensions = [".txt", ".xml", ".json"]
    found = {}
    for ext in extensions:
        files = list(base_dir.rglob(f"*{ext}"))
        if files:
            found[ext] = len(files)
    return found

# annot_types = find_annotation_files(FLAME_DIR)
# print("Annotation formats found:", annot_types)

## 5. Next Steps

1. **Masks:** made from the thermal frames by `src/embereye/data/flame2.py`, and checked against 48 hand-labelled frames (`docs/findings/flame2-label-validation.md`).
2. **Split:** contiguous blocks of frames, **not** a random 80/10/10 split. Neighbouring frames are near-copies, so a random split leaks test frames into training (`src/embereye/data/flame2_split.py`).
3. **Export:** `python scripts/export_flame2_yolo.py` writes YOLO-seg labels and the `data.yaml` files to `data/flame2/yolo_seg.nosync/`; notebook `02_generate_annotations.ipynb` shows single frames.
4. **Train** YOLO11-seg (milestone M4). There is no training notebook yet.